# Aviation Accidents AnalysisYou are part of a consulting firm tasked with analyzing aviation safety. This notebook contains the data cleaning pipeline.

### Make relevant library imports

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.- inspect NaNs, datatypes, and summary statistics

In [ ]:
df = pd.read_csv('../../aviationData/AviationData.csv', encoding='latin-1', low_memory=False)print('Dataset shape:', df.shape)print('Columns:', df.columns.tolist())print('\nData types:')print(df.dtypes)

## Data Cleaning

### Filtering aircrafts and events

In [ ]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')max_date = df['Event.Date'].max()cutoff_date = max_date - pd.DateOffset(years=40)print(f'Date range: {df["Event.Date"].min().date()} to {max_date.date()}')print(f'Filtering for events >= {cutoff_date.date()}')df = df[df['Aircraft.Category'] == 'Airplane'].copy()df = df[df['Amateur.Built'] == 'No'].copy()df = df[df['Event.Date'] >= cutoff_date].copy()print(f'After filtering: {len(df)} rows')

### Cleaning and constructing Key Measurables

**Construct metric for fatal/serious injuries**

In [ ]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']for col in injury_cols:    df[col] = df[col].fillna(0)df['Total.Passengers'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + df['Total.Minor.Injuries'] + df['Total.Uninjured']df['Fatal.Serious.Injury.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / (df['Total.Passengers'] + 1)df['Fatal.Serious.Injury.Fraction'] = df['Fatal.Serious.Injury.Fraction'].fillna(0).clip(0, 1)print('Injury metrics created')print(df['Fatal.Serious.Injury.Fraction'].describe())

**Aircraft.Damage**

In [ ]:
df['Aircraft.Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)df['Aircraft.Destroyed'] = df['Aircraft.Destroyed'].fillna(-1)df['Aircraft.Destroyed_Clean'] = df['Aircraft.Destroyed'].copy()df.loc[df['Aircraft.Destroyed_Clean'] == -1, 'Aircraft.Destroyed_Clean'] = np.nanprint('Aircraft destruction metrics created')print(df['Aircraft.Destroyed'].value_counts())

### Investigate the *Make* column

In [ ]:
df = df[df['Make'].notna()].copy()make_counts = df['Make'].value_counts()makes_to_keep = make_counts[make_counts >= 50].indexdf = df[df['Make'].isin(makes_to_keep)].copy()print(f'Filtered to {len(makes_to_keep)} makes with >= 50 accidents')print(f'Dataset shape: {len(df)} rows')

### Inspect Model column

In [ ]:
df = df[df['Model'].notna()].copy()df['Aircraft.Type'] = df['Make'] + ' ' + df['Model']print(f'Created {df["Aircraft.Type"].nunique()} unique aircraft types')

### Cleaning other columns

In [ ]:
df = df[df['Number.of.Engines'] > 0].copy()df['Engine.Type'] = df['Engine.Type'].fillna('Unknown')df['Purpose.of.flight'] = df['Purpose.of.flight'].fillna('Unknown')df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')print('Other columns cleaned')print(f'Final dataset shape: {df.shape}')

### Column Removal

In [ ]:
non_null_counts = df.isnull().apply(lambda x: (x == False).sum())threshold = 20000cols_to_drop = non_null_counts[non_null_counts < threshold].index.tolist()df = df.drop(columns=cols_to_drop)print(f'Columns remaining: {len(df.columns)}')print(f'Final shape: {df.shape}')

### Save DataFrame to csv

In [ ]:
output_path = '../../data/aviation_cleaned.csv'df.to_csv(output_path, index=False)print(f'Cleaned data saved to: {output_path}')print(f'Dataset statistics:')print(f'  Rows: {len(df)}')print(f'  Columns: {len(df.columns)}')